# Notebook tái lập kết quả và phân tích thống kê

Notebook này có hai chế độ. Mặc định, notebook **không tiền xử lý dữ liệu và không huấn luyện lại mô hình**: nó dùng các tệp điểm đã có trong `scores/`, kiểm tra các đầu vào cần cho phân tích, rồi chạy `results/final-stats.ipynb` để tính lại kiểm định, bảng và kết luận.

## Chế độ mặc định: tính lại thống kê từ tệp điểm đã lưu

- `RUN_EXPERIMENTS=0`: bỏ qua các bước tạo điểm, gồm kiểm tra/thí nghiệm mô hình và các notebook huấn luyện. Tệp `scores/*.npy` hiện có được giữ nguyên và dùng làm đầu vào.
- `RUN_FINAL_STATS=1`: chạy lại `results/final-stats.ipynb`; xuất `results/final_stats.csv`, `results/final_stats_tables_with_notes.csv`, `results/verdicts.json` và manifest của các tệp điểm.
- Không chạy tiền xử lý thô `src/data/01-combine-data.ipynb` hoặc tạo lại split bằng `src/data/02-data-splits.ipynb`. Dữ liệu đã xử lý, embedding và tệp điểm được dùng lại.
- Cần checkout đầy đủ cùng dữ liệu Git LFS. Không cần GPU để chỉ tính lại thống kê.

## Nếu muốn tái chạy từ dữ liệu thô

Trước tiên chạy `src/data/01-combine-data.ipynb` rồi `src/data/02-data-splits.ipynb` bằng dữ liệu nguồn thích hợp. Sau đó, nếu muốn tạo lại hồ sơ/embedding LLM, đặt `REGENERATE_LLM_PROFILES=1` (cần GPU NVIDIA tương thích); đặt `RUN_EXPERIMENTS=1` để chạy lại các kiểm tra và mô hình tạo tệp điểm; giữ `RUN_FINAL_STATS=1` để tính lại thống kê sau cùng. Việc chạy đầy đủ có thể mất nhiều giờ.

CGRC bị loại khỏi quy trình tái lập và khỏi các bảng tổng hợp. USIM chỉ có bước sàng lọc raw tùy chọn, một seed, không tinh chỉnh. Kết quả đã có đạt hai điều kiện đầu nhưng không đạt điều kiện NDCG@20 cold-test: 0.0053 so với mốc LightGCN 0.0165. Vì vậy báo cáo USIM như một hạn chế; không chạy fused/LLM hay tinh chỉnh. Chỉ đặt `RUN_USIM_RAW_SCREEN=1` nếu muốn lặp lại phép sàng lọc đó.

Muốn tái lập đúng phiên bản đã công bố, đặt `REPRO_REPO_REF` thành commit SHA 40 ký tự. SHA thực tế được ghi vào tệp kê khai. Nếu đã có bản checkout đầy đủ, đặt `REPRO_REPO_PATH`; nếu không, notebook cần Internet để sao chép mã và tải dữ liệu Git LFS.


In [ ]:
import importlib.util
import os
import platform
import sys
from pathlib import Path

REPO_URL = os.environ.get(
    "REPRO_REPO_URL",
    "https://github.com/datachain-uit/CN-23520430-HanLTN-DiemLK.git",
)
REPO_REF = os.environ.get("REPRO_REPO_REF", "draft")
REPO_NAME = "CN-23520430-HanLTN-DiemLK"
SEED = int(os.environ.get("REPRO_SEED", "2026"))

INSTALL_DEPENDENCIES = True
# Mặc định dùng tệp dữ liệu/điểm đã lưu; không chạy tiền xử lý hoặc huấn luyện.
# Đặt RUN_EXPERIMENTS=1 để chạy lại khối thí nghiệm ở ô cuối.
# Đặt RUN_FINAL_STATS=0 nếu chỉ muốn kiểm tra môi trường/tệp đầu vào.
# Kaggle và Colab giữ các phiên bản thư viện do nền tảng quản lý, trừ khi bật ghim nghiêm ngặt.
# Chỉ bật khi muốn tạo lại hồ sơ LLM đã lưu (cần GPU tương thích).
REGENERATE_LLM_PROFILES = os.environ.get("REGENERATE_LLM_PROFILES", "0").strip().lower() in {"1", "true", "yes"}
RUN_EXPERIMENTS = os.environ.get("RUN_EXPERIMENTS", "0").strip().lower() not in {"0", "false", "no"}
RUN_FINAL_STATS = os.environ.get("RUN_FINAL_STATS", "1").strip().lower() not in {"0", "false", "no"}
RUN_USIM_RAW_SCREEN = os.environ.get("RUN_USIM_RAW_SCREEN", "0").strip().lower() in {"1", "true", "yes"}


def detect_runtime():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"):
        return "Kaggle"
    try:
        import google.colab  # noqa: F401
        return "Colab"
    except Exception:
        pass
    if os.environ.get("VSCODE_PID") or os.environ.get("TERM_PROGRAM") == "vscode":
        return "Máy cá nhân / VS Code"
    return f"Máy cá nhân / Jupyter ({platform.system()})"


RUNTIME = detect_runtime()
_default_strict = "0" if RUNTIME in {"Kaggle", "Colab"} else "1"
STRICT_PINS = os.environ.get("REPRO_STRICT_PINS", _default_strict).strip().lower() in {"1", "true", "yes"}
print(f"Môi trường: {RUNTIME} | Python: {platform.python_version()} | Hệ điều hành: {platform.platform()}")
print(
    f"Tạo lại hồ sơ LLM: {REGENERATE_LLM_PROFILES} | "
    f"Chạy lại thí nghiệm/tạo tệp điểm: {RUN_EXPERIMENTS} | "
    f"Chạy final stats trên tệp điểm hiện có: {RUN_FINAL_STATS} | "
    f"Sàng lọc USIM raw: {RUN_USIM_RAW_SCREEN}"
)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path


def find_repository(start: Path):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (
            (candidate / "README.md").is_file()
            and (candidate / "data").is_dir()
            and (candidate / "src").is_dir()
        ):
            return candidate
    return None


def run_command(args, *, cwd=None, env=None):
    args = [str(arg) for arg in args]
    print("\n$", subprocess.list2cmdline(args) if os.name == "nt" else " ".join(args))
    return subprocess.run(args, cwd=cwd, env=env, check=True, text=True)


def ensure_git_lfs():
    try:
        probe = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
    except FileNotFoundError as exc:
        raise RuntimeError("Không tìm thấy Git. Hãy cài Git rồi khởi động lại kernel.") from exc
    if probe.returncode == 0:
        return True

    # Kaggle và Colab cho phép cài Git LFS qua apt; các môi trường khác nhận hướng dẫn cụ thể.
    if RUNTIME in {"Kaggle", "Colab"} and shutil.which("apt-get"):
        for command in (["apt-get", "update", "-qq"], ["apt-get", "install", "-y", "-qq", "git-lfs"]):
            result = subprocess.run(command, capture_output=True, text=True)
            if result.returncode != 0:
                details = (result.stdout + "\n" + result.stderr).strip()
                raise RuntimeError(f"Không cài được Git LFS: {' '.join(command)}\n{details[-2000:]}")
        probe = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
        if probe.returncode == 0:
            print(probe.stdout.strip())
            return True

    print(
        "Git LFS chưa khả dụng. Tiếp tục kiểm tra; nếu checkout còn tệp LFS dạng pointer, "
        "bước kiểm tra dữ liệu sẽ báo những tệp cần tải."
    )
    return False


configured_root = os.environ.get("REPRO_REPO_PATH")
if configured_root:
    REPO_ROOT = Path(configured_root).expanduser().resolve()
    if not ((REPO_ROOT / "README.md").is_file() and (REPO_ROOT / "data").is_dir() and (REPO_ROOT / "src").is_dir()):
        raise FileNotFoundError(f"REPRO_REPO_PATH không trỏ tới gốc repository hợp lệ: {REPO_ROOT}")
else:
    REPO_ROOT = find_repository(Path.cwd())
    if REPO_ROOT is None:
        clone_dir = Path.cwd().resolve() / "workspace" / REPO_NAME
        clone_dir.parent.mkdir(parents=True, exist_ok=True)
        if clone_dir.exists():
            raise FileExistsError(
                f"Thư mục sao chép đã tồn tại nhưng không phải repository hợp lệ: {clone_dir}. "
                "Đặt REPRO_REPO_PATH tới checkout hợp lệ hoặc chọn thư mục làm việc khác."
            )
        if len(REPO_REF) == 40 and all(ch in "0123456789abcdefABCDEF" for ch in REPO_REF):
            run_command(["git", "clone", REPO_URL, clone_dir])
            run_command(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=clone_dir)
            run_command(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=clone_dir)
        else:
            run_command(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, clone_dir])
        REPO_ROOT = clone_dir.resolve()

os.environ["DATA_DIR"] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Thư mục gốc repository: {REPO_ROOT}")
if (REPO_ROOT / ".git").exists():
    if ensure_git_lfs():
        run_command(["git", "lfs", "install"], cwd=REPO_ROOT)
        run_command(["git", "lfs", "pull"], cwd=REPO_ROOT)
    REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
else:
    print("Không tìm thấy thư mục .git; giả định checkout đã có đủ dữ liệu.")
    REPO_COMMIT = "không có thông tin (bản xuất repository)"
print(f"Phiên bản repository được chọn: {REPO_COMMIT}")

In [ ]:
import importlib.metadata as metadata
import json
import os
import random
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

LOCK_FILE = REPO_ROOT / "requirements.txt"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = REPO_ROOT / "results" / "reproduction_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)


def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def locked_packages(path):
    packages = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or line.startswith("-"):
            continue
        match = re.match(r"^([A-Za-z0-9_.-]+)==([^;\s]+)", line)
        if match:
            packages.append((match.group(1), match.group(2)))
    return packages


def pip_install(specs, extra_args=()):
    if not specs:
        return
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed"]
    command.extend(extra_args)
    command.extend(specs)
    run_command(command, cwd=REPO_ROOT)


package_status = {}
if INSTALL_DEPENDENCIES:
    if not LOCK_FILE.is_file():
        raise FileNotFoundError(f"Không tìm thấy tệp khóa phiên bản thư viện phụ thuộc: {LOCK_FILE}")
    missing_or_mismatched = []
    host_mismatches = []
    for name, version in locked_packages(LOCK_FILE):
        if name.lower() == "bitsandbytes" and not REGENERATE_LLM_PROFILES:
            package_status[name] = installed_version(name) or "chưa cài (thư viện tùy chọn để tạo hồ sơ LLM)"
            continue
        if name.lower() == "torch":
            continue  # Cài đặt hoặc kiểm tra PyTorch riêng để chọn bản phù hợp với nền tảng.
        current = installed_version(name)
        package_status[name] = current
        if current == version:
            continue
        if current is not None and RUNTIME in {"Kaggle", "Colab"} and not STRICT_PINS:
            host_mismatches.append((name, current, version))
        else:
            missing_or_mismatched.append(f"{name}=={version}")

    if missing_or_mismatched:
        pip_install(missing_or_mismatched)
    if host_mismatches:
        print("\nĐang dùng các phiên bản thư viện do nền tảng cung cấp để tránh làm hỏng môi trường notebook được quản lý:")
        for name, current, expected in host_mismatches:
            print(f"  {name}: phiên bản đã cài {current}; phiên bản thí nghiệm yêu cầu {expected}")
        print("Để áp dụng chính xác các phiên bản đã ghim, hãy dùng môi trường cục bộ mới hoặc đặt REPRO_STRICT_PINS=1.")

    torch_pin = next((v for n, v in locked_packages(LOCK_FILE) if n.lower() == "torch"), "2.11.0")
    torch_base = torch_pin.split("+", 1)[0]
    torch_current = installed_version("torch")
    package_status["torch"] = torch_current
    has_nvidia = shutil.which("nvidia-smi") is not None
    current_cuda_build = bool(torch_current and "+cu128" in torch_current)
    should_replace_torch = torch_current is None or (
        STRICT_PINS
        and (torch_current.split("+", 1)[0] != torch_base or (has_nvidia and not current_cuda_build))
    )
    if should_replace_torch:
        if has_nvidia:
            pip_install([f"torch=={torch_pin}"], ["--extra-index-url", "https://download.pytorch.org/whl/cu128"])
        elif platform.system() in {"Windows", "Linux"}:
            pip_install([f"torch=={torch_base}"], ["--index-url", "https://download.pytorch.org/whl/cpu"])
        else:
            pip_install([f"torch=={torch_base}"])
    if REGENERATE_LLM_PROFILES:
        bnb_current = installed_version("bitsandbytes")
        bnb_pin = next((v for n, v in locked_packages(LOCK_FILE) if n.lower() == "bitsandbytes"), None)
        if bnb_pin and bnb_current != bnb_pin:
            pip_install([f"bitsandbytes=={bnb_pin}"])
    # nbconvert không nằm trong danh sách thư viện của thí nghiệm nhưng cần để chạy các notebook thành phần.
    nbconvert_version = installed_version("nbconvert")
    if nbconvert_version is None or tuple(int(x) for x in nbconvert_version.split(".")[:2]) < (7, 16):
        pip_install(["nbconvert>=7.16,<8"])
else:
    print("Đã tắt cài đặt thư viện phụ thuộc; notebook sẽ tiếp tục dùng môi trường kernel hiện tại.")

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
try:
    import numpy as np
    np.random.seed(SEED)
except ImportError:
    np = None
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except (AttributeError, TypeError):
        pass
except ImportError:
    torch = None
try:
    import tensorflow as tf
    tf.random.set_seed(SEED)
except Exception as exc:
    tf = None
    print(f"Bỏ qua việc cố định seed cho TensorFlow ({type(exc).__name__}: {exc})")

# Ghi lại phiên bản thực tế (cùng phiên bản đã ghim) để nhận biết khác biệt giữa các môi trường.
for package, _ in locked_packages(LOCK_FILE):
    package_status[package] = installed_version(package)
for package in ["numpy", "pandas", "pyarrow", "scipy", "sentence-transformers", "transformers", "accelerate", "bitsandbytes", "torch", "nbconvert"]:
    package_status[package] = installed_version(package)
manifest = {
    "run_id": RUN_ID,
    "runtime": RUNTIME,
    "os": platform.platform(),
    "python": platform.python_version(),
    "seed": SEED,
    "run_experiments": RUN_EXPERIMENTS,
    "run_final_stats": RUN_FINAL_STATS,
    "run_usim_raw_screen": RUN_USIM_RAW_SCREEN,
    "repository": REPO_URL,
    "requested_ref": REPO_REF,
    "resolved_commit": REPO_COMMIT,
    "strict_pins": STRICT_PINS,
    "packages": package_status,
    "torch_cuda_available": bool(torch is not None and torch.cuda.is_available()),
    "torch_cuda_version": None if torch is None else torch.version.cuda,
}
(RUN_DIR / "environment_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Đã cố định seed ở {SEED}; tệp kê khai môi trường được lưu tại {RUN_DIR / 'environment_manifest.json'}")
if torch is not None:
    print(f"PyTorch: {torch.__version__} | Có CUDA: {torch.cuda.is_available()} | Số thiết bị: {torch.cuda.device_count()}")

In [ ]:
from pathlib import Path

# Tệp cần cho final-stats và ColdRecDataAdapter để kiểm tra dữ liệu/nhóm người học.
# Đây là kiểm tra đầu vào đã lưu, không tạo lại dữ liệu hoặc embedding.
required_files = [
    Path("data/embeddings/01_lightgcn_backbone_cold/item_index.json"),
    Path("data/embeddings/01_lightgcn_backbone_cold/user_index_cold.json"),
    Path("data/embeddings/01_lightgcn_backbone_cold/g_u.npy"),
    Path("data/embeddings/01_lightgcn_backbone_cold/g_i.npy"),
    Path("data/splits/cold_scenario/cold_items_val.json"),
    Path("data/splits/cold_scenario/cold_items_test.json"),
    Path("data/splits/cold_scenario/train_cold.parquet"),
    Path("data/splits/cold_scenario/val_cold.parquet"),
    Path("data/splits/cold_scenario/test_cold.parquet"),
    Path("data/course_llm/course_dup_groups.parquet"),
]

if not REGENERATE_LLM_PROFILES:
    required_files.extend([
        Path("data/course_llm/1_embeddings/raw_course_text_embeddings.npy"),
        Path("data/course_llm/1_embeddings/llm_profile_embeddings.npy"),
        Path("data/course_llm/1_embeddings/llm_profile_no_prereq_followup_embeddings.npy"),
    ])
else:
    # Đầu vào cần trước khi chạy bước tạo profile tùy chọn.
    required_files.append(Path("data/course_llm/course_data.parquet"))

if RUN_EXPERIMENTS:
    # Các thí nghiệm cần thêm dữ liệu đã xử lý. Các notebook tiền xử lý dữ liệu thô
    # được nêu ở ô mở đầu; chúng không chạy tự động trong chế độ mặc định.
    required_files.extend([
        Path("data/full_interactions/interactions_full.parquet"),
        Path("data/raw_data_processed/user-course.csv"),
        Path("data/course_llm/course_data.parquet"),
    ])

if RUN_FINAL_STATS and not RUN_EXPERIMENTS:
    # Xác nhận các tệp điểm đã lưu trước khi tính lại thống kê, tránh chạy final-stats
    # rồi mới phát hiện checkout thiếu score.
    score_dirs = {
        "knn": "Content-KNN",
        "dropoutnet": "DropoutNet",
        "aldi": "ALDI",
        "semproj": "SemProj",
    }
    score_inputs = {
        "knn": ["raw", "llm", "fused"],
        "dropoutnet": ["raw", "llm", "fused", "llm_3d", "fused_3d"],
        "aldi": ["raw", "llm", "fused", "llm_3d", "fused_3d", "llm_robust3s"],
        "semproj": ["raw", "llm", "fused", "llm_3d", "fused_3d"],
    }
    for model, inputs in score_inputs.items():
        for input_name in inputs:
            required_files.append(
                Path("scores") / score_dirs[model] / f"user_ndcg20_{model}_{input_name}.npy"
            )
        for input_name in ("raw", "fused"):
            for group in ("moi", "trung"):
                required_files.append(
                    Path("scores") / score_dirs[model] / f"user_ndcg20_{model}_{input_name}_{group}.npy"
                )

missing_files = [str(path) for path in required_files if not (REPO_ROOT / path).is_file()]
lfs_pointers = []
for relative in required_files:
    path = REPO_ROOT / relative
    if path.is_file():
        try:
            with path.open("rb") as file:
                if file.read(120).startswith(b"version https://git-lfs.github.com/spec/v1"):
                    lfs_pointers.append(str(relative))
        except OSError:
            pass
if missing_files or lfs_pointers:
    details = []
    if missing_files:
        details.append("Thiếu tệp đầu vào của chế độ đã chọn: " + ", ".join(missing_files))
    if lfs_pointers:
        details.append("Chưa tải nội dung Git LFS: " + ", ".join(lfs_pointers))
    raise FileNotFoundError(
        "\n".join(details)
        + "\nHãy dùng checkout đầy đủ/dữ liệu Git LFS. Nếu tái chạy từ dữ liệu thô, hãy chạy các notebook tiền xử lý được ghi ở ô đầu tiên trước."
    )
print(f"Kiểm tra thành công {len(required_files)} tệp đầu vào của chế độ đã chọn.")


In [ ]:
import shutil
from pathlib import Path


def run_python(*args):
    command = [sys.executable, *map(str, args)]
    env = os.environ.copy()
    env["DATA_DIR"] = str(REPO_ROOT)
    env["PYTHONHASHSEED"] = str(SEED)
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    run_command(command, cwd=REPO_ROOT, env=env)


if REGENERATE_LLM_PROFILES:
    if torch is None or not torch.cuda.is_available():
        raise RuntimeError(
            "Tạo lại hồ sơ Qwen 7B theo quy trình 4-bit đã ghim phiên bản cần GPU NVIDIA hỗ trợ CUDA. "
            "Hãy dùng hồ sơ đã lưu (mặc định) hoặc bật GPU tương thích trước khi đặt REGENERATE_LLM_PROFILES=True."
        )
    profile_dir = Path("data/course_llm/2_profiles_full_and_shards")
    for shard in (0, 1):
        run_python(
            "llm/course_profile.py",
            "--data_path", "data/course_llm/course_data.parquet",
            "--prompt_path", "src/prompts/prompt_template.txt",
            "--output_dir", profile_dir,
            "--num_shards", "2",
            "--shard", str(shard),
        )
    run_python(
        "llm/course_profile.py",
        "--data_path", "data/course_llm/course_data.parquet",
        "--output_dir", profile_dir,
        "--num_shards", "2",
        "--merge_shards",
    )
    embedding_dir = REPO_ROOT / "data/course_llm/1_embeddings"
    embedding_dir.mkdir(parents=True, exist_ok=True)
    for artifact in list((REPO_ROOT / profile_dir).glob("*embeddings*.npy")) + list((REPO_ROOT / profile_dir).glob("*course_index.json")):
        shutil.copy2(artifact, embedding_dir / artifact.name)
    run_python(
        "llm/course_profile.py",
        "--data_path", "data/course_llm/course_data.parquet",
        "--prompt_path", "src/prompts/prompt_template.txt",
        "--output_dir", "data/course_llm/1_embeddings",
        "--encode_raw_only",
    )
    run_python(
        "llm/course_profile.py",
        "--data_path", "data/course_llm/course_data.parquet",
        "--prompt_path", "src/prompts/prompt_template.txt",
        "--output_dir", "data/course_llm/3_quality_evaluation",
        "--masked_prereq_eval",
    )
else:
    print("Đang dùng các hồ sơ và embedding có sẵn trong repository hoặc được cung cấp qua LFS; bỏ qua bước tạo hồ sơ.")

In [ ]:
import hashlib
import subprocess
from pathlib import Path


def run_script(relative_path):
    path = REPO_ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy script trong quy trình: {path}")
    run_python(path)


def run_notebook(relative_path):
    source = REPO_ROOT / relative_path
    if not source.is_file():
        raise FileNotFoundError(f"Không tìm thấy notebook trong quy trình: {source}")
    command = [
        sys.executable, "-m", "nbconvert",
        "--to", "notebook", "--execute",
        "--ExecutePreprocessor.timeout=-1",
        "--output", f"{source.stem}-executed.ipynb",
        "--output-dir", str(RUN_DIR),
        str(source),
    ]
    env = os.environ.copy()
    env["DATA_DIR"] = str(REPO_ROOT)
    env["PYTHONHASHSEED"] = str(SEED)
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    run_command(command, cwd=REPO_ROOT, env=env)


if RUN_EXPERIMENTS:
    # Chế độ đầy đủ: chạy lại kiểm tra và các notebook/scripts tạo tệp điểm.
    print("Đang chạy lại kiểm tra, thí nghiệm mô hình và tạo tệp điểm...")
    for item in (
        "src/eval/check_cold_items.py",
        "src/eval/random_ndcg_analytic.py",
        "src/eval/time_consistent_subset.py",
        "src/models/coldrec_adapter.py",
        "src/models/run_aldi.py",
    ):
        print(f"\n=== Đang chạy {item} ===")
        run_script(item)

    backbone_files = [
        REPO_ROOT / "data/embeddings/01_lightgcn_backbone_cold/g_u.npy",
        REPO_ROOT / "data/embeddings/01_lightgcn_backbone_cold/g_i.npy",
    ]
    for path in backbone_files:
        if path.is_file():
            digest = hashlib.md5(path.read_bytes()).hexdigest()
            print(f"MD5 {path.relative_to(REPO_ROOT)}: {digest}")

    core_notebooks = [
        "src/models/lightgcn_leakage.ipynb",
        "src/models/cold_scenario/lightgcn_backbone_cold.ipynb",
        "src/eval/evaluate_test_random.ipynb",
        "src/models/cold_scenario/lightgcn_id_cold.ipynb",
        "src/models/cold_scenario/content-knn-v2.ipynb",
        "src/models/cold_scenario/dropoutnet-v2.ipynb",
        "src/models/cold_scenario/dropoutnet-v2-p-rq3.ipynb",
        "src/models/cold_scenario/dropoutnet-llm-3d.ipynb",
        "src/models/semproj_experiments.ipynb",
    ]
    for item in core_notebooks:
        print(f"\n=== Đang chạy {item} ===")
        run_notebook(item)

else:
    print(
        "\nĐang dùng lại các tệp điểm .npy đã lưu. Bỏ qua tiền xử lý dữ liệu và toàn bộ khối kiểm tra/huấn luyện ở trên. "
        "Muốn chạy lại các thí nghiệm tạo điểm, đặt RUN_EXPERIMENTS=1; muốn chạy từ dữ liệu thô, xem hướng dẫn ở ô đầu tiên."
    )

if RUN_USIM_RAW_SCREEN:
    print(
        "\n=== Sàng lọc USIM raw: một lần chạy, một seed; "
        "không tinh chỉnh và không chạy fused/LLM ==="
    )
    run_notebook("src/models/coldrec-adapter-usim-raw.ipynb")
else:
    print(
        "\nBỏ qua USIM theo mặc định. Nếu cần xem lại phép sàng lọc raw duy nhất, "
        "đặt RUN_USIM_RAW_SCREEN=1."
    )

if RUN_FINAL_STATS:
    print("\n=== Đang chạy results/final-stats.ipynb trên các tệp điểm hiện có/đã tạo lại ===")
    run_notebook("results/final-stats.ipynb")
    print(f"\nĐã hoàn tất tính lại thống kê. Notebook thực thi được lưu tại: {RUN_DIR}")
else:
    print("Đã bỏ qua final-stats. Đặt RUN_FINAL_STATS=1 để tính lại thống kê từ các tệp điểm.")